# 第 12 课作业：处理一个 source spike 的完整旅程

这份作业对应 **第 12 课：一个 spike 的完整旅程**。

前面已经分别学过 queue、source index、synapse record 和 target accumulator。现在只处理 **一个 source spike**，把 lookup、weighted event 和 target update 串成一条清楚的因果链。

## 本题 contract

process_one_spike() 接收 source_id、source_index、records 和 target_accum，并且：

1. 找到该 source 的 (start, count)；
2. 只遍历对应 range；
3. 每条 (target, weight) 产生一个同样的 weighted event；
4. 把 weight 加到对应 target accumulator；
5. 返回新的 accumulator 与 weighted_events；
6. 不原地修改传入的 target_accum。

starter 已经提供 updated = list(target_accum)。这是有意提供的 scaffold：本题主要考 event routing 与 target update，不考 Python list copy 技巧。

## 先不用代码：手工追踪一个 source range

给定：

- source_index = [(0, 1), (1, 2), (3, 0)]
- records = [(2, +4), (0, -2), (2, +3)]
- target_accum = [5, 0, 1]
- source_id = 1

先回答：

1. source 1 的 start/count 是多少？
2. 它允许读取 records 的哪一段？
3. weighted_events 应按什么顺序产生？
4. 每个 event 发生前后，对应 target accumulator 怎样变化？
5. source 0 的 record 在这次调用里为什么绝对不能被处理？

最后再检查原始 target_accum 是否应该保持原内容不变。

## 实现 process_one_spike()

### 这个函数做什么？

`process_one_spike()` 表示系统收到 **一个 source spike** 以后，从 sparse lookup 一直到 target accumulation 的完整软件模型。

本函数只处理这一个 source 的 outgoing synapses。

### 输入

- `source_id`：这次发生 spike 的 source neuron；
- `source_index`：每个 source 对应的 `(start, count)`；
- `records`：连续保存的 `(target, weight)` synapse records；
- `target_accum`：处理这个 spike **之前**，所有 target 当前已经累积的输入值。

starter 已经先执行 `updated = list(target_accum)`，因此你应当修改 `updated`，而不是原地修改传入的 `target_accum`。

### 输出

函数返回 **两个值**，顺序固定为：

`(updated, weighted_events)`

其中：

1. `updated`：一个新的 target accumulator 列表。  
   对当前 source range 中的每条 `(target, weight)`，把 `weight` 加到对应的 `updated[target]`；其他 target 保持原值。
2. `weighted_events`：一个列表，记录这次 source spike 实际产生的所有 weighted events。  
   每一项是 `(target, weight)`，顺序应与当前 source 在 `records` 中的顺序一致。

如果当前 source 的 `count = 0`：

- `weighted_events` 应为空列表；
- `updated` 的数值内容应与原 `target_accum` 相同，但仍然是一个新的列表对象。

这里的 `weighted_events` 描述“这次 spike 要把哪些权重送到哪些 target”，而 `updated` 描述“这些 event 应用以后留下的累积状态”。

In [ ]:
def process_one_spike(
    source_id: int,
    source_index: list[tuple[int, int]],
    records: list[tuple[int, int]],
    target_accum: list[int],
) -> tuple[list[int], list[tuple[int, int]]]:
    updated = list(target_accum)

    # YOUR CODE STARTS HERE
    raise NotImplementedError("TODO: lookup range and apply weighted events")
    # YOUR CODE ENDS HERE

    return updated, weighted_events

## 检查你的实现

grader 会检查 routing、source isolation、zero fanout 与“不修改输入 accumulator”的 API 契约，不显示具体测试向量。

In [ ]:
# Course infrastructure: make the repository root importable from a notebook subdirectory.
from pathlib import Path
import sys

_repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "exercises" / "grader").is_dir()
)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from exercises.grader.lesson12 import check

check(process_one_spike=process_one_spike, language="zh")

## Human Check

不用看 grader，指着你的代码说明：

1. source_index 在哪一步把 source_id 变成一个 records range？
2. synapse record 与 weighted event 在本题里分别表示什么？
3. target accumulator 保存的是什么，为什么它不是 spike queue？
4. count=0 时应该产生多少 weighted event，updated accumulator 应怎样变化？
5. 为什么“一次 source spike 只处理自己的 range”是 event-driven 计算的关键？